# Add `is_misdemeanor` and `has_warrant` Columns

Adds two derived columns to checkpoint14 and saves checkpoint15.

- **`is_misdemeanor`**: `True` if every charge in the row classifies as `Misdemeanor` under MA law; `False` if any charge is `Felony`, `Either`, or `Unknown`; `NaN` for rows with no charges.
- **`has_warrant`**: `True` if any individual charge in the original `Charges` field carried a warrant prefix (bench warrant, default warrant, standard warrant, capias, child in need); `False` otherwise; `NaN` for rows with no charges.

**Input:** `data/checkpoints/checkpoint14_standardized_charges.csv`  
**Output:** `data/checkpoints/checkpoint15_misdemeanor_warrant.csv`

### Imports & Paths

In [8]:
import os
import numpy as np
import pandas as pd

from standardize_charges import extract_warrant_type

NOTEBOOK_DIR = os.getcwd()

DATA_DIR = os.path.join(
    NOTEBOOK_DIR,
    "..",
    "data",
    "missing_dates_csv",
)

CHECKPOINTS_DIR = os.path.join(DATA_DIR, "md_checkpoints")

IN_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint14_standardized_charges.csv",
)

OUT_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint15_misdemeanor_warrant.csv",
)


REPO_ROOT = os.path.abspath(
    os.path.join(NOTEBOOK_DIR, "..", "..")
)

LOOKUP_PATH = os.path.join(
    REPO_ROOT,
    "streamlit-app",
    "unique_charges_standardized.csv",
)

### Load checkpoint14

In [15]:
df = pd.read_csv(IN_PATH, low_memory=False)
display(df.head())
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Rows with charges: {df['cleaned_charges'].notna().sum()}")
df.head()

,Incident #,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,latitude,longitude,geocode_confidence,person_id,category,Year,crime_severity,Age,statutes,cleaned_charges
0,18000001.0,2018-01-01 00:01:14,NOISE ORD,3 HARRIMAN ST,Yes,NaN,1974-07-03,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,3 HARRIMAN ST,42.718522,-71.148148,10.0,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,Public Disturbances,2018,Non-Serious,43.0,NaN,NaN
1,18000002.0,2018-01-01 00:08:38,LOUD NOISE,1 HARRIMAN ST,Yes,NaN,1979-01-21,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,1 HARRIMAN ST FL 2,42.718522,-71.148148,10.0,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,Public Disturbances,2018,Non-Serious,38.0,NaN,NaN
2,18000003.0,2018-01-01 00:11:17,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,16 ALLEN ST,42.710782,-71.151911,10.0,NaN,Fire and Arson Incidents,2018,Non-Serious,NaN,NaN,NaN
3,18000004.0,2018-01-01 00:14:53,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,11 SUMMER ST,42.711117,-71.153015,10.0,NaN,Public Disturbances,2018,Non-Serious,NaN,NaN,NaN
4,18000005.0,2018-01-01 00:27:36,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,2002-06-26,A&B DOMESTIC NO 209A IN EFFECT,57 SPRINGFIELD ST,42.699340,-71.156938,10.0,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,Preventive Policing,2018,Non-Serious,15.0,NaN,NaN


Shape: (428527, 19)
Columns: ['Incident #', 'Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'latitude', 'longitude', 'geocode_confidence', 'person_id', 'category', 'Year', 'crime_severity', 'Age', 'statutes', 'cleaned_charges']
Rows with charges: 0


,Incident #,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,latitude,longitude,geocode_confidence,person_id,category,Year,crime_severity,Age,statutes,cleaned_charges
0,18000001.0,2018-01-01 00:01:14,NOISE ORD,3 HARRIMAN ST,Yes,NaN,1974-07-03,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,3 HARRIMAN ST,42.718522,-71.148148,10.0,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,Public Disturbances,2018,Non-Serious,43.0,NaN,NaN
1,18000002.0,2018-01-01 00:08:38,LOUD NOISE,1 HARRIMAN ST,Yes,NaN,1979-01-21,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,1 HARRIMAN ST FL 2,42.718522,-71.148148,10.0,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,Public Disturbances,2018,Non-Serious,38.0,NaN,NaN
2,18000003.0,2018-01-01 00:11:17,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,16 ALLEN ST,42.710782,-71.151911,10.0,NaN,Fire and Arson Incidents,2018,Non-Serious,NaN,NaN,NaN
3,18000004.0,2018-01-01 00:14:53,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,11 SUMMER ST,42.711117,-71.153015,10.0,NaN,Public Disturbances,2018,Non-Serious,NaN,NaN,NaN
4,18000005.0,2018-01-01 00:27:36,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,2002-06-26,A&B DOMESTIC NO 209A IN EFFECT,57 SPRINGFIELD ST,42.699340,-71.156938,10.0,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,Preventive Policing,2018,Non-Serious,15.0,NaN,NaN


### Build charge_class lookup

Load `unique_charges_standardized.csv` and build a dict: `base_charge → charge_class`.

In [16]:
#gffi
lookup_df = pd.read_csv(LOOKUP_PATH)

charge_class_lookup = dict(
    zip(
        lookup_df["base_charge"],
        lookup_df["charge_class"]
    )
)

print(f"Lookup entries: {len(charge_class_lookup)}")
print(f"Classes present: {sorted(lookup_df['charge_class'].dropna().unique())}")

print()
for sample in [
    "trespass",
    "murder",
    "firearm, carry without license",
    "drug, possess class a",
]:
    print(f"{sample!r} -> {charge_class_lookup.get(sample, 'NOT FOUND')}")

Lookup entries: 394
Classes present: ['Either', 'Felony', 'Misdemeanor']

'trespass' -> Misdemeanor
'murder' -> Felony
'firearm, carry without license' -> Either
'drug, possess class a' -> Misdemeanor


In [17]:
 
lookup_df = pd.read_csv(LOOKUP_PATH)
charge_class_lookup = dict(zip(lookup_df["base_charge"], lookup_df["charge_class"]))

print(f"Lookup entries: {len(charge_class_lookup)}")
print(f"Classes present: {set(charge_class_lookup.values())}")

# Quick sanity checks
print()
for sample in ["trespass", "murder", "firearm, carry without license", "drug, possess class a"]:
    print(f"  {sample!r} -> {charge_class_lookup.get(sample, 'NOT FOUND')}")

Lookup entries: 394
Classes present: {'Felony', 'Misdemeanor', 'Either'}

  'trespass' -> Misdemeanor
  'murder' -> Felony
  'firearm, carry without license' -> Either
  'drug, possess class a' -> Misdemeanor


### Compute `has_warrant`

Check the original `Charges` column. Split each row's charges by `;` and run `extract_warrant_type()` on each individual charge. If any charge had a warrant prefix, the row is flagged `True`.

In [18]:
## old 
def row_has_warrant(raw_charges):
    """Return True if any individual charge in the raw Charges string
    carries a warrant prefix. Returns NaN for rows with no charges."""
    if pd.isna(raw_charges):
        return np.nan
    parts = [p.strip() for p in str(raw_charges).split(";") if p.strip()]
    for part in parts:
        warrant_type, _ = extract_warrant_type(part)
        if warrant_type != "none":
            return True
    return False


df["has_warrant"] = df["cleaned_charges"].apply(row_has_warrant)

warrant_counts = df["has_warrant"].value_counts(dropna=False)
print("has_warrant value counts:")
print(warrant_counts)

# Spot-check
print("\nSample warrant rows:")
sample_warrant = df[df["has_warrant"] == True][["Charges", "cleaned_charges", "has_warrant"]].head(5)
for _, row in sample_warrant.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['cleaned_charges']}")
    print()

has_warrant value counts:
has_warrant
NaN    428527
Name: count, dtype: int64

Sample warrant rows:


### Compute `is_misdemeanor`

Split `standardized_charges` by `"; "` and look up each charge's class in the lookup dict. A row is `True` only when **every** charge in the row classifies as `Misdemeanor`. Any `Felony`, `Either`, or `Unknown` charge makes the row `False`. Rows with no charges return `NaN`.

In [19]:
#gffi 
def row_is_misdemeanor(std_charges):
    """Return True if every standardized charge in the row is Misdemeanor.
    Returns False if any charge is Felony, Either, or Unknown.
    Returns NaN for rows with no charges."""
    if pd.isna(std_charges):
        return np.nan
    parts = [p.strip() for p in str(std_charges).split(";") if p.strip()]
    if not parts:
        return np.nan
    for charge in parts:
        cls = charge_class_lookup.get(charge, "Unknown")
        if cls != "Misdemeanor":
            return False
    return True


df["is_misdemeanor"] = df["cleaned_charges"].apply(row_is_misdemeanor)

misdemeanor_counts = df["is_misdemeanor"].value_counts(dropna=False)
print("is_misdemeanor value counts:")
print(misdemeanor_counts)

# Spot-check True rows
print("\nSample misdemeanor-only rows:")
sample_misd = df[df["is_misdemeanor"] == True][["Charges", "cleaned_charges", "is_misdemeanor"]].head(5)
for _, row in sample_misd.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['cleaned_charges']}")
    print()

# Spot-check False rows (not pure misdemeanor)
print("Sample non-misdemeanor rows:")
sample_non = df[df["is_misdemeanor"] == False][["Charges", "cleaned_charges", "is_misdemeanor"]].head(5)
for _, row in sample_non.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['cleaned_charges']}")
    print()

is_misdemeanor value counts:
is_misdemeanor
NaN    428527
Name: count, dtype: int64

Sample misdemeanor-only rows:
Sample non-misdemeanor rows:


### Diagnose any charges not found in lookup

Identifies standardized charges that appear in the data but are missing from `unique_charges_standardized.csv`.

In [20]:
all_std = (
    df["cleaned_charges"]
    .dropna()
    .str.split("; ")
    .explode()
    .str.strip()
    .unique()
)

missing = [c for c in all_std if c and c not in charge_class_lookup]
if missing:
    print(f"{len(missing)} standardized charges not found in lookup:")
    for c in sorted(missing):
        print(f"  {c!r}")
else:
    print("All cleaned charges found in lookup.")

AttributeError: Can only use .str accessor with string values!

### Verify & summarize

In [22]:
charged_rows = df[df["cleaned_charges"].notna()]

print(f"Total rows: {len(df):,}")
print(f"Rows with charges: {len(charged_rows):,}")
print()
print(f"has_warrant = True  : {(df['has_warrant'] == True).sum():,}")
print(f"has_warrant = False : {(df['has_warrant'] == False).sum():,}")
print(f"has_warrant = NaN   : {df['has_warrant'].isna().sum():,}")
print()
print(f"is_misdemeanor = True  : {(df['is_misdemeanor'] == True).sum():,}")
print(f"is_misdemeanor = False : {(df['is_misdemeanor'] == False).sum():,}")
print(f"is_misdemeanor = NaN   : {df['is_misdemeanor'].isna().sum():,}")
print()
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Total rows: 428,527
Rows with charges: 0

has_warrant = True  : 0
has_warrant = False : 0
has_warrant = NaN   : 428,527

is_misdemeanor = True  : 0
is_misdemeanor = False : 0
is_misdemeanor = NaN   : 428,527

Final shape: (428527, 21)
Columns: ['Incident #', 'Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'latitude', 'longitude', 'geocode_confidence', 'person_id', 'category', 'Year', 'crime_severity', 'Age', 'statutes', 'cleaned_charges', 'has_warrant', 'is_misdemeanor']


### Save checkpoint15

In [23]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved checkpoint15: {OUT_PATH}")
print(f"Shape: {df.shape}")

Saved checkpoint15: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/charges/../data/missing_dates_csv/md_checkpoints/checkpoint15_misdemeanor_warrant.csv
Shape: (428527, 21)


In [24]:
with pd.option_context('display.max_rows', None):
    print(df['Location'].value_counts())

Location
90 LOWELL ST                                    4811
BRADFORD ST & BROADWAY                          3043
700 ESSEX ST                                    2170
50 BROADWAY                                     1930
266 BROADWAY                                    1768
BROADWAY & LOWELL ST                            1690
73 WINTHROP AV                                  1686
S UNION ST & WINTHROP AV                        1652
205 BROADWAY                                    1631
BROADWAY & ESSEX ST                             1609
BROADWAY & CROSS ST                             1590
240 CANAL ST                                    1450
BROADWAY & HAVERHILL ST                         1405
370 BROADWAY                                    1372
1 GENERAL ST                                    1273
BROADWAY & WATER ST                             1140
300 CANAL ST                                    1107
MERRIMACK ST & S UNION ST                       1081
HAVERHILL ST & JACKSON ST            